<a href="https://colab.research.google.com/github/Srinithi-A/SRIML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-05 — Feature Vector and Leakage/Privacy Check

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Srinithi-A/SRIML/blob/main/work/notebooks/w03_feature_leakage_check.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Build the feature vector

*Code that actually builds it — engineered features, categorical handling, fills.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [4]:
import pandas as pd

# Load dataset
df = pd.read_csv("https://raw.githubusercontent.com/Srinithi-A/SRIML/refs/heads/main/data/raw/content_refresh_anonymized.csv")

# Select features
features = [
    "avg_position",
    "impressions_90d",
    "ctr",
    "engagement_rate",
    "content_age_days"
]

X = df[features].copy()

# Fill missing values
X = X.fillna(0)

print("Feature Matrix Shape:", X.shape)

X.head()

Feature Matrix Shape: (30000, 5)


,avg_position,impressions_90d,ctr,engagement_rate,content_age_days
0,10.6,3803,0.76,5.88,187
1,20.3,15320,0.05,0.00,445
2,36.5,12581,0.09,0.00,141
3,6.2,11751,0.49,1.28,463
4,44.0,19140,0.13,0.00,263


# 1. Build the Feature Vector

The feature vector contains only information that is available before making an optimization decision. These features describe the historical search performance and content characteristics of each webpage.

The selected features are:
- Average search position
- Impressions (90 days)
- Click-through rate (CTR)
- Engagement rate
- Content age (days)

Missing values are filled using appropriate defaults, and categorical variables are encoded before modeling.

## 2. Feature notes (meaning, missing, categorical, available-when?)

*For each feature: what it means, how missing values are handled, and whether it exists BEFORE the moment you predict.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 2. Feature Notes

| Feature | Meaning | Missing Values | Available Before Prediction? |
|---------|---------|----------------|------------------------------|
| avg_position | Average search position | Filled with 0 if missing | Yes |
| impressions_90d | Search impressions over the past 90 days | Filled with 0 | Yes |
| ctr | Historical click-through rate | Filled with 0 | Yes |
| engagement_rate | User engagement metric | Filled with 0 | Yes |
| content_age_days | Age of the webpage in days | Filled with 0 | Yes |

All selected features are historical values and are available before making the optimization decision.

In [6]:
print(X.info())

print("\nMissing Values:")
print(X.isnull().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 5 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   avg_position      30000 non-null  float64
 1   impressions_90d   30000 non-null  int64  
 2   ctr               30000 non-null  float64
 3   engagement_rate   30000 non-null  float64
 4   content_age_days  30000 non-null  int64  
dtypes: float64(3), int64(2)
memory usage: 1.1 MB
None

Missing Values:
avg_position        0
impressions_90d     0
ctr                 0
engagement_rate     0
content_age_days    0
dtype: int64


## 3. The leakage hunt

*Attack your own features: label-derived columns, future windows, product flags. Show the test.*

In [7]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 3. The Leakage Hunt

I checked whether any feature directly contains information about the target or future performance.

Examples of leakage include:
- Future CTR
- Future clicks
- Product-generated flags
- Label-derived columns

No such fields are included in the final feature vector.

In [8]:
# Example proxy target
df["needs_optimization"] = (df["ctr"] < 0.05).astype(int)

# Deliberately create a leakage feature
df["leak_feature"] = df["needs_optimization"]

print(df[["needs_optimization", "leak_feature"]].head())

# Remove the leakage feature
df.drop(columns=["leak_feature"], inplace=True)

print("Leakage feature removed successfully.")

   needs_optimization  leak_feature
0                   0             0
1                   0             0
2                   0             0
3                   0             0
4                   0             0
Leakage feature removed successfully.


## 4. What I excluded and why

*The list of fields you refused to use — with one line of why each.*

In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


# 4. What I Excluded and Why

| Excluded Field | Reason |
|---------------|--------|
| Future CTR | Not available at prediction time |
| Future Clicks | Uses future information |
| Product Flags | Derived from internal business logic rather than raw inputs |
| Label Columns | Would reveal the correct answer to the model |
| Client Identifiers | Not useful for prediction and should remain private |

These fields were excluded to prevent data leakage and protect privacy.

In [10]:
excluded_fields = [
    "future_ctr",
    "future_clicks",
    "product_flag",
    "label",
    "client_id"
]

print("Excluded fields:")
for field in excluded_fields:
    print("-", field)

Excluded fields:
- future_ctr
- future_clicks
- product_flag
- label
- client_id


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.